# MeViS — смешанные видео (человек + животные/объекты)
**Цель:** отобрать видео из смешанных категорий, ранжированных по упоминанию человека.  
**Фильтр промтов:** только те, где ровно один объект (`len(obj_id) == 1`).  
**Viewer:** 5 кадров на видео (0% / 25% / 50% / 75% / 100%) + список single-object промтов.

In [2]:
import json
import os
import tarfile

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

BASE       = "/home/jovyan/shares/SR006.nfs2/vashina/datasets/MeViS/data_sets/train"
JSON_PATH  = f"{BASE}/meta_expressions_v2.json"
JPEG_DIR   = f"{BASE}/JPEGImages_fequency/JPEGImages"   # уже распакованные 300 видео
TAR_PATH   = f"{BASE}/JPEGImages.tar"                   # архив со всеми 1662
OUTPUT_TXT = f"{BASE}/liked_videos_mixed.txt"

HUMAN_KW  = ['person','man','woman','people','rider','girl','boy','child','human',
              'men','women','hand','hands']
ANIMAL_KW = ['cat','dog','bird','rabbit','zebra','tiger','monkey','bear','fish',
              'giraffe','elephant','sheep','panda','lizard','horse','cow','duck',
              'deer','fox','wolf','lion','cheetah','leopard','gorilla','frog',
              'turtle','snake','parrot','eagle','penguin','seal','whale','dolphin',
              'camel','goat','pig','chicken','kangaroo','koala','crocodile',
              'squirrel','hamster','mouse','rat','avian','creature','animal',
              'beast','cub','pup','kitten','puppy']
OBJECT_KW = ['car','truck','bus','vehicle','motorcycle','bicycle','bike','boat',
              'ship','plane','airplane','train','ball','balloon']

with open(JSON_PATH) as f:
    data = json.load(f)

print(f"Всего видео в датасете: {len(data['videos'])}")
print(f"Уже распакованных папок: {len(os.listdir(JPEG_DIR))}")

Всего видео в датасете: 1662
Уже распакованных папок: 313


## 1. Формируем таблицу смешанных видео

In [3]:
rows = []
for vid_id, vdata in data['videos'].items():
    exps = vdata['expressions']
    all_texts = [e['exp'].lower() for e in exps.values()]

    has_human  = any(any(k in t for k in HUMAN_KW)  for t in all_texts)
    has_animal = any(any(k in t for k in ANIMAL_KW) for t in all_texts)
    has_object = any(any(k in t for k in OBJECT_KW) for t in all_texts)

    # только смешанные: человек + (животное или объект)
    if not (has_human and (has_animal or has_object)):
        continue

    human_cnt  = sum(any(k in t for k in HUMAN_KW)  for t in all_texts)
    total_exp  = len(exps)
    n_frames   = len(vdata['frames'])

    # single-object промты: ровно один obj_id
    single_prompts = {
        eid: edata['exp']
        for eid, edata in exps.items()
        if len(edata['obj_id']) == 1
    }

    if not single_prompts:
        continue

    if   has_animal and not has_object: group = 'human+animal'
    elif has_object and not has_animal: group = 'human+object'
    else:                               group = 'human+animal+object'

    extracted = os.path.isdir(os.path.join(JPEG_DIR, vid_id))

    rows.append({
        'vid_id':         vid_id,
        'group':          group,
        'human_cnt':      human_cnt,
        'total_exp':      total_exp,
        'n_single':       len(single_prompts),
        'n_frames':       n_frames,
        'extracted':      extracted,
        'single_prompts': single_prompts,   # dict {exp_id: text}
    })

df = pd.DataFrame(rows)

# сортировка: по группе (alphabetical), внутри — по human_cnt убывающе
GROUP_ORDER = {'human+animal': 0, 'human+object': 1, 'human+animal+object': 2}
df['_g'] = df['group'].map(GROUP_ORDER)
df = df.sort_values(['_g', 'human_cnt'], ascending=[True, False]).drop(columns='_g').reset_index(drop=True)

print(f"Смешанных видео с single-obj промтами: {len(df)}")
print()
for g, sub in df.groupby('group', sort=False):
    extr = sub['extracted'].sum()
    print(f"  {g:30s}: {len(sub):4d} видео  (уже распакованных: {extr})")

Смешанных видео с single-obj промтами: 925

  human+animal                  :  316 видео  (уже распакованных: 34)
  human+object                  :  340 видео  (уже распакованных: 77)
  human+animal+object           :  269 видео  (уже распакованных: 36)


## 2. Сводная таблица — топ-20 по каждой группе

In [4]:
def show_top(group_name, n=20):
    sub = df[df['group'] == group_name].head(n).copy()
    sub = sub[['vid_id','human_cnt','total_exp','n_single','n_frames','extracted']]
    sub.index = range(1, len(sub)+1)
    return sub

print("=== human + animal ===")
display(show_top('human+animal'))
print("\n=== human + object ===")
display(show_top('human+object'))
print("\n=== human + animal + object ===")
display(show_top('human+animal+object'))

=== human + animal ===


,vid_id,human_cnt,total_exp,n_single,n_frames,extracted
1,b7d7eae3d3b0,41,61,59,37,True
2,46bd43bd1cd1,35,45,38,500,True
3,d17e414f7caf,35,52,37,180,True
4,e0542977cb49,30,47,37,35,True
5,3f5b9db67a69,29,31,28,90,True
6,5a0679487ac8,28,44,29,240,False
7,050d15dc37b1,26,48,29,180,False
8,db498351c1b4,26,43,22,113,False
9,66d2732b3fee,25,37,7,32,False
10,a69be20f6223,24,35,18,30,False



=== human + object ===


,vid_id,human_cnt,total_exp,n_single,n_frames,extracted
1,be280cdfcc4b,48,96,65,75,True
2,e09e09dbbca6,36,56,32,71,True
3,75018886fd35,31,68,51,34,True
4,23c436e47472,28,53,37,35,True
5,5c8c047d1136,27,30,28,90,True
6,952f0ee83a37,26,53,48,143,False
7,1c1789cba8e0,26,53,52,48,False
8,496f3127d149,25,46,28,150,False
9,e4edace38a55,24,50,15,40,False
10,5b3a72d53378,24,50,32,31,False



=== human + animal + object ===


,vid_id,human_cnt,total_exp,n_single,n_frames,extracted
1,197b9dd9a0dd,57,91,67,34,True
2,6bda0dd3b0a8,54,109,99,113,True
3,ce734b61fa1b,51,119,90,25,True
4,92bee97eb8cc,50,95,72,89,True
5,ad322268bf9f,47,77,75,36,True
6,70e522047ad8,39,93,82,42,False
7,7b17095cb301,38,51,37,240,False
8,9e072d4cf000,36,53,48,60,False
9,1812b773db69,35,48,7,30,False
10,f645fd6c15ae,34,113,95,43,False


## Предпросмотр: топ-5 видео группы `human+animal+object`
Выводим 5 кадров (0 % / 25 % / 50 % / 75 % / 100 %) и single-object промты.  
Если видео не распаковано — оно извлекается из `.tar` автоматически.

In [ ]:
import tarfile

def ensure_frames(vid_id: str) -> str | None:
    """Возвращает путь к папке с кадрами, распаковывая из tar при необходимости."""
    dst = os.path.join(JPEG_DIR, vid_id)
    if os.path.isdir(dst) and os.listdir(dst):
        return dst
    prefix = f"JPEGImages/{vid_id}/"
    try:
        with tarfile.open(TAR_PATH) as tar:
            members = [m for m in tar.getmembers() if m.name.startswith(prefix)]
            if not members:
                return None
            for m in members:
                rel = m.name[len("JPEGImages/"):]
                out_path = os.path.join(JPEG_DIR, rel)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                with tar.extractfile(m) as src_f, open(out_path, 'wb') as dst_f:
                    dst_f.write(src_f.read())
        return dst
    except Exception as e:
        print(f"  Ошибка распаковки {vid_id}: {e}")
        return None


def preview_video(row):
    vid_id        = row['vid_id']
    single_prompts = row['single_prompts']

    frame_dir = ensure_frames(vid_id)
    if frame_dir is None:
        print(f"⚠️  Кадры для {vid_id} недоступны")
        return

    frames = sorted(os.listdir(frame_dir))
    n = len(frames)
    indices  = [0, n // 4, n // 2, 3 * n // 4, n - 1]
    selected = [frames[i] for i in indices]
    labels   = ["0%", "25%", "50%", "75%", "100%"]

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(
        f"{vid_id}  │  [{row['group']}]  │  human={int(row['human_cnt'])}/{int(row['total_exp'])}  │  "
        f"single-obj промтов: {int(row['n_single'])}  │  кадров: {int(row['n_frames'])}",
        fontsize=10
    )
    for ax, fname, label in zip(axes, selected, labels):
        img = mpimg.imread(os.path.join(frame_dir, fname))
        ax.imshow(img)
        ax.set_title(f"{label}  {fname}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"  Single-object промты ({len(single_prompts)}):")
    for eid, text in sorted(single_prompts.items(), key=lambda x: int(x[0])):
        print(f"    [{eid}] {text}")
    print()


# топ-5 видео группы human+animal+object
top5 = df[df['group'] == 'human+animal+object'].head(5).to_dict('records')
for row in top5:
    preview_video(row)

In [ ]:
## Предпросмотр: топ-5 видео группы `human+object`

In [ ]:
top5_obj = df[df['group'] == 'human+object'].head(5).to_dict('records')
for row in top5_obj:
    preview_video(row)

In [ ]:
## Предпросмотр: топ-5 видео группы `human+animal`

In [ ]:
top5_animal = df[df['group'] == 'human+animal'].head(5).to_dict('records')
for row in top5_animal:
    preview_video(row)

## 3. Создание датасета FirstMetricsData

Формируем три папки по 100 видео:
- **humans_frequency** — топ-100 видео с людьми, ранжированных по частоте упоминания (`human_cnt / total_exp`)
- **human_object** — топ-100 из группы `human+object` (по `human_cnt`)
- **human_animal_object** — топ-100 из группы `human+animal+object` (по `human_cnt`)

Если видео ещё не распаковано из `JPEGImages.tar` — распаковывается автоматически.

In [5]:
import shutil
import subprocess

# ─── 1. Топ-100 из jpgimages_frequency ────────────────────────────────────────
KEYWORDS_FREQ = ["person", "man", "woman", "people", "rider", "girl",
                 "boy", "child", "human"]

freq_records = []
for vid_id, vdata in data["videos"].items():
    exps = [e["exp"].lower() for e in vdata["expressions"].values()]
    total = len(exps)
    human_cnt = sum(any(kw in exp for kw in KEYWORDS_FREQ) for exp in exps)
    frequency = human_cnt / total if total > 0 else 0.0
    freq_records.append({"vid_id": vid_id, "human_cnt": human_cnt,
                          "total_exp": total, "frequency": frequency})

df_freq = pd.DataFrame(freq_records)
df_freq = df_freq[df_freq["human_cnt"] > 0].sort_values(
    ["frequency", "human_cnt"], ascending=False
).reset_index(drop=True)
top100_freq = df_freq.head(100)["vid_id"].tolist()

# ─── 2. Топ-100 из human+object и human+animal+object ─────────────────────────
top100_obj      = df[df["group"] == "human+object"].head(100)["vid_id"].tolist()
top100_anim_obj = df[df["group"] == "human+animal+object"].head(100)["vid_id"].tolist()

print(f"humans_frequency   : {len(top100_freq)} видео")
print(f"human_object       : {len(top100_obj)} видео")
print(f"human_animal_object: {len(top100_anim_obj)} видео")

# ─── 3. Создаём структуру папок ───────────────────────────────────────────────
FIRST_METRICS_DIR = os.path.join(BASE, "FirstMetricsData")
SUBSETS = {
    "humans_frequency":    top100_freq,
    "human_object":        top100_obj,
    "human_animal_object": top100_anim_obj,
}
for name in SUBSETS:
    os.makedirs(os.path.join(FIRST_METRICS_DIR, name), exist_ok=True)
print(f"\nПапка датасета: {FIRST_METRICS_DIR}")

# ─── 4. Вспомогательные функции ───────────────────────────────────────────────
ALREADY_EXTRACTED = JPEG_DIR  # BASE/JPEGImages_fequency/JPEGImages
TMP_DIR = os.path.join(BASE, "_tmp_extract")

def find_extracted(vid_id: str):
    """Возвращает путь к уже распакованной папке или None."""
    p = os.path.join(ALREADY_EXTRACTED, vid_id)
    return p if os.path.isdir(p) else None

def extract_from_tar(vid_ids: list, dst_dir: str):
    """Распаковывает список vid_id из JPEGImages.tar в dst_dir/JPEGImages/."""
    os.makedirs(dst_dir, exist_ok=True)
    paths_in_tar = [f"JPEGImages/{v}" for v in vid_ids]
    cmd = ["tar", "-xf", TAR_PATH, "-C", dst_dir] + paths_in_tar
    subprocess.run(cmd, check=True)

# ─── 5. Копируем видео в FirstMetricsData ─────────────────────────────────────
for subset_name, vid_ids in SUBSETS.items():
    dst_subset = os.path.join(FIRST_METRICS_DIR, subset_name)
    to_extract = [v for v in vid_ids if find_extracted(v) is None
                  and not os.path.isdir(os.path.join(TMP_DIR, "JPEGImages", v))]

    if to_extract:
        print(f"[{subset_name}] Распаковываем из tar: {len(to_extract)} видео...")
        extract_from_tar(to_extract, TMP_DIR)

    copied = skipped = 0
    for vid_id in vid_ids:
        dst = os.path.join(dst_subset, vid_id)
        if os.path.exists(dst):
            skipped += 1
            continue
        src = find_extracted(vid_id)
        if src is None:
            src = os.path.join(TMP_DIR, "JPEGImages", vid_id)
        shutil.copytree(src, dst)
        copied += 1

    print(f"[{subset_name}] скопировано: {copied}, уже было: {skipped}")

# ─── 6. Удаляем временную папку ───────────────────────────────────────────────
if os.path.exists(TMP_DIR):
    shutil.rmtree(TMP_DIR)
    print("\nВременная папка _tmp_extract удалена.")

# ─── 7. Итоговая статистика ───────────────────────────────────────────────────
print("\n=== Итог ===")
for subset_name in SUBSETS:
    path = os.path.join(FIRST_METRICS_DIR, subset_name)
    n = len(os.listdir(path))
    print(f"  {subset_name:25s}: {n} видео  →  {path}")

humans_frequency   : 100 видео
human_object       : 100 видео
human_animal_object: 100 видео

Папка датасета: /home/jovyan/shares/SR006.nfs2/vashina/datasets/MeViS/data_sets/train/FirstMetricsData
[humans_frequency] скопировано: 100, уже было: 0
[human_object] Распаковываем из tar: 59 видео...
[human_object] скопировано: 100, уже было: 0
[human_animal_object] Распаковываем из tar: 77 видео...
[human_animal_object] скопировано: 100, уже было: 0

Временная папка _tmp_extract удалена.

=== Итог ===
  humans_frequency         : 100 видео  →  /home/jovyan/shares/SR006.nfs2/vashina/datasets/MeViS/data_sets/train/FirstMetricsData/humans_frequency
  human_object             : 100 видео  →  /home/jovyan/shares/SR006.nfs2/vashina/datasets/MeViS/data_sets/train/FirstMetricsData/human_object
  human_animal_object      : 100 видео  →  /home/jovyan/shares/SR006.nfs2/vashina/datasets/MeViS/data_sets/train/FirstMetricsData/human_animal_object
